<a href="https://colab.research.google.com/github/ryougishikifor214/torchcode/blob/master/templates/14_kv_cache.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/14_kv_cache.ipynb)

# 🔴 Hard: KV Cache Attention

Implement **multi-head attention with KV caching** for efficient autoregressive generation.

During LLM inference, recomputing all key/value projections at every step is wasteful.
A **KV cache** stores previously computed K and V tensors so only the new token(s) need projection.

### Signature
```python
class KVCacheAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, x: torch.Tensor, cache=None) -> tuple[torch.Tensor, tuple]:
        # x: (B, S_new, D) — new tokens
        # cache: None or (K_past, V_past) each (B, num_heads, S_past, d_k)
        # Returns: (output, (K_all, V_all))
```

### Requirements
- Inherit from `nn.Module`
- `self.W_q`, `self.W_k`, `self.W_v`, `self.W_o`: `nn.Linear` projections
- When `cache=None` (prefill): apply **causal mask**, return all K/V as cache
- When `cache` provided (decode): concat new K/V with cached, no causal mask needed for single-token decode
- Incremental decode must produce **identical** results to full forward pass

### Key Idea
```
Prefill:  [t0 t1 t2 t3] → full causal attention → cache = (K_{0:3}, V_{0:3})
Decode:   [t4]           → Q=t4, K/V=cache+t4  → cache = (K_{0:4}, V_{0:4})
Decode:   [t5]           → Q=t5, K/V=cache+t5  → cache = (K_{0:5}, V_{0:5})
```

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.2 MB/s eta 0:00:00


In [5]:
import torch
import torch.nn as nn
import math
from torch_judge import hint
hint("kv_cache")


💡 Hint for KV Cache Attention:
   Project Q/K/V, reshape to (B, num_heads, S, d_k). If cache exists, concat new K/V with cached along dim=2. Apply causal mask during prefill. Return (output, (K_all, V_all)). Cache tensors: (B, num_heads, S_total, d_k).



In [10]:
# ✏️ YOUR IMPLEMENTATION HERE

class KVCacheAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        # pass  # Initialize W_q, W_k, W_v, W_o
        self.d_model=d_model
        self.num_heads=num_heads
        self.d_k=d_model//num_heads
        self.W_q=nn.Linear(d_model,d_model)
        self.W_k=nn.Linear(d_model,d_model)
        self.W_v=nn.Linear(d_model,d_model)
        self.W_o=nn.Linear(d_model,d_model)

    def forward(self, x, cache=None):
        # 1. Project Q, K, V from x
        # 2. Reshape to multi-head: (B, num_heads, S, d_k)
        # 3. If cache exists, concat new K/V with cached K/V
        # 4. Compute attention (causal mask needed during prefill)
        # 5. Return (output, (K_all, V_all))
        # pass
        B,S_new,_=x.shape
        Q,K,V=self.W_q(x),self.W_k(x),self.W_v(x)
        Q=Q.view(B,S_new,self.num_heads,self.d_k).permute(0,2,1,3)
        K=K.view(B,S_new,self.num_heads,self.d_k).permute(0,2,1,3)
        V=V.view(B,S_new,self.num_heads,self.d_k).permute(0,2,1,3)

        if cache:
          print(cache)
          K=torch.cat([cache[0],K],dim=2)
          V=torch.cat([cache[1],V],dim=2)

        new_cache=(K,V)
        S_total=K.shape[2]
        scores=torch.matmul(Q,K.transpose(-1,-2))/math.sqrt(self.d_k)
        if S_new>1:
          S_past=S_total-S_new
          mask=torch.triu(
              torch.ones(S_new,S_total,device=x.device,dtype=torch.bool),
              diagonal=S_past+1,
          )
          scores=scores.masked_fill(mask,float("-inf"))

        weights=torch.softmax(scores,dim=-1)
        attn=weights@V
        attn=attn.permute(0,2,1,3).contiguous().view(B,S_new,-1)
        output=self.W_o(attn)
        return output,new_cache



In [11]:
# 🧪 Debug
torch.manual_seed(0)
attn = KVCacheAttention(d_model=64, num_heads=4)
x = torch.randn(1, 6, 64)

# Full forward
full_out, _ = attn(x)
print("Full output shape:", full_out.shape)  # (1, 6, 64)

# Incremental: prefill 4, decode 1, decode 1
out1, cache = attn(x[:, :4])
print("Cache K shape:", cache[0].shape)  # (1, 4, 4, 16)
out2, cache = attn(x[:, 4:5], cache=cache)
out3, cache = attn(x[:, 5:6], cache=cache)
inc_out = torch.cat([out1, out2, out3], dim=1)
print("Match:", torch.allclose(full_out, inc_out, atol=1e-5))

Full output shape: torch.Size([1, 6, 64])
Cache K shape: torch.Size([1, 4, 4, 16])
(tensor([[[[ 3.4246e-01, -2.0102e-01,  4.5755e-01, -1.9780e-01,  1.5623e-02,
            3.6512e-01,  8.8590e-01,  1.9239e-01, -8.8977e-01,  2.5716e-01,
           -5.8635e-01,  5.3452e-01,  9.1256e-02, -4.4025e-01,  2.5874e-03,
           -1.4410e+00],
          [-3.7915e-01,  5.5893e-03,  5.4145e-01,  1.2659e-01,  2.1324e-01,
           -7.8179e-01, -5.9200e-02, -1.0985e+00, -2.5042e-02, -1.3406e-01,
            4.0627e-01,  9.6422e-01, -1.5594e-01, -1.8587e-03,  2.2881e-01,
           -9.4054e-01],
          [ 2.2097e-01, -4.6514e-01, -2.2594e-01,  3.6934e-01, -8.3764e-01,
            3.1501e-02,  5.9693e-01,  6.2347e-02,  2.2840e-01, -1.9750e-01,
           -2.4961e-01, -9.1361e-01,  6.1612e-02, -6.1132e-01,  6.6927e-01,
           -4.1194e-01],
          [-3.9725e-01, -9.9970e-01,  8.2329e-01,  1.0722e+00,  8.0384e-01,
           -5.7212e-01,  3.0841e-01, -3.9726e-01, -4.5119e-01,  3.0341e-01,
     

In [12]:
# ✅ SUBMIT
from torch_judge import check
check('kv_cache')


🧪 Testing: KV Cache Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/5] Output shape (no cache) (4.2ms)
  ✅ [2/5] Cache structure (1.5ms)
(tensor([[[[-0.3148,  0.1735, -0.5685,  0.8065, -1.1846, -0.3022,  1.3289,
            0.0195, -0.5933,  1.1876, -0.2374, -0.0198,  1.1922, -1.3720,
           -0.1658, -0.8508],
          [ 0.3551, -0.0620,  0.6935,  0.0407,  0.4133, -0.2539, -0.1939,
            0.1604,  0.2512,  0.2456, -0.2946,  0.3464,  1.0200,  0.2146,
           -0.2699, -0.1530],
          [ 0.9572, -0.0345,  0.6981,  1.0963, -0.3396, -0.1619, -0.8097,
            0.4962, -0.6880, -0.3464,  0.0386, -0.2460, -0.3483, -0.0027,
           -0.2159, -0.3125],
          [-0.5260, -1.0058, -1.1543, -0.3570,  1.1444, -0.3408, -1.1315,
            0.1867, -0.1232, -0.5987,  0.4708, -0.0298, -1.0614, -0.1325,
           -0.5562,  1.7816]],

         [[-0.1995, -0.3018, -1.0235, -0.2827,  0.0031, -0.4968, -0.1888,
            0.1824, -0.8993,  0.4087,  0.7971,